# Week 4 — DP variants playground

Open in VS Code. **Select Kernel** (Python 3). Run from the top.

Three tables from the lecture, each on its toy:

1. **Hirschberg** on `ACATATG` against `ACTATA`: the score from two columns, the crossing of the middle column, then the full alignment.
2. **Nussinov** on `GGCC`: the most non-crossing pairs.
3. **Viterbi** and the forward sum on the two-state island machine.

## Part 1 · Hirschberg

Lecture scores: match $+1$, mismatch $-1$, gap $-1$. `v` runs down the rows, `w` across the columns.

### Idea 1 · The score needs two columns

Sweep column by column. Each new column reads only the one before it, so we keep two lists, never the grid.

In [ ]:
v = "ACATATG"   # down the rows
w = "ACTATA"    # across the columns
GAP = -1

def sigma(a, b):
    return 1 if a == b else -1

def last_column(v, w):
    """Scores of every prefix of v against all of w, using two columns."""
    col = [GAP * i for i in range(len(v) + 1)]          # column 0: only gaps
    for j in range(1, len(w) + 1):
        new = [GAP * j]
        for i in range(1, len(v) + 1):
            new.append(max(col[i - 1] + sigma(v[i - 1], w[j - 1]),   # diagonal
                           col[i] + GAP,                              # right
                           new[i - 1] + GAP))                         # down
        col = new
    return col

score = last_column(v, w)[-1]
print("best score:", score)
assert score == 3

### Idea 2 · Every path crosses the middle column

$F(i)$: best score of the first $i$ letters of `v` against the left half `ACT`.
$B(i)$: best score of the rest of `v` against the right half `ATA` — the same sweep on both strings reversed.
A path through row $i$ of the middle column scores $F(i)+B(i)$. The largest sum is where the best path crosses.

In [ ]:
mid = len(w) // 2
F = last_column(v, w[:mid])
B = last_column(v[::-1], w[mid:][::-1])[::-1]
sums = [f + b for f, b in zip(F, B)]

print("row  F   B  F+B")
for i, (f, b, s) in enumerate(zip(F, B, sums)):
    print(f"{i:>3} {f:>3} {b:>3} {s:>4}")

row = sums.index(max(sums))
print("crossing in row", row)
assert F == [-3, -1, 1, 1, 2, 1, 0, -1]
assert B == [-1, 0, 1, 0, 1, -1, -3, -3]
assert row == 4 and max(sums) == score

### Idea 3 · Repeat on the two pieces

Cut at the crossing, solve the top-left and bottom-right pieces the same way. A piece one column wide is aligned directly.

In [ ]:
def one_column(v, c):
    """Align v against the single letter c: put c under its best partner, gaps elsewhere."""
    if not v:
        return "-", c
    best = max(range(len(v)), key=lambda i: sigma(v[i], c))
    if sigma(v[best], c) + GAP * (len(v) - 1) < GAP * (len(v) + 1):
        return v + "-", "-" * len(v) + c
    return v, "-" * best + c + "-" * (len(v) - best - 1)

def hirschberg(v, w):
    if not w:
        return v, "-" * len(v)
    if len(w) == 1:
        return one_column(v, w)
    mid = len(w) // 2
    F = last_column(v, w[:mid])
    B = last_column(v[::-1], w[mid:][::-1])[::-1]
    sums = [f + b for f, b in zip(F, B)]
    i = sums.index(max(sums))
    top_v, top_w = hirschberg(v[:i], w[:mid])
    bot_v, bot_w = hirschberg(v[i:], w[mid:])
    return top_v + bot_v, top_w + bot_w

def column_score(a, b):
    return sum(GAP if "-" in (x, y) else sigma(x, y) for x, y in zip(a, b))

a, b = hirschberg(v, w)
print(a)
print(b)
print("score:", column_score(a, b))
assert (a, b) == ("ACATATG", "AC-TATA")

**Play.** Try two longer strings, for example `v = "GATTACAGATTACA"` and `w = "GACTACAGTTACA"`. Check that `column_score(*hirschberg(v, w))` equals `last_column(v, w)[-1]`.

## Part 2 · Nussinov on `GGCC`

Cell $N[i][j]$ is the most non-crossing pairs in the substring from $i$ to $j$ (inclusive).
Pairs: A–U, G–C. Fill short substrings first: one diagonal at a time, toward the top-right corner.

In [ ]:
seq = "GGCC"
n = len(seq)
pair = {('G','C'),('C','G'),('A','U'),('U','A')}

N = [[0] * n for _ in range(n)]     # the diagonal (one base) stays 0

def can_pair(i, j):
    return (seq[i], seq[j]) in pair

for length in range(1, n):          # j = i + length: one diagonal per length
    for i in range(n - length):
        j = i + length
        best = max(N[i + 1][j], N[i][j - 1])                     # drop i, drop j
        if can_pair(i, j):
            inner = N[i + 1][j - 1] if i + 1 <= j - 1 else 0
            best = max(best, inner + 1)                          # pair i with j
        for k in range(i, j):
            best = max(best, N[i][k] + N[k + 1][j])              # split after k
        N[i][j] = best

print("Nussinov table for", seq)
for i, row in enumerate(N):
    print(["." if j < i else x for j, x in enumerate(row)])
print("whole molecule N[0][n-1] =", N[0][n - 1])
assert N[0][n - 1] == 2

**Play.** Try `seq = "GGAUCC"` or `"GGCCGG"`. Does $N[0][n-1]$ match what you expect by hand?

Optional: the [Freiburg Nussinov playground](https://rna.informatik.uni-freiburg.de/Teaching/index.jsp?toolName=Nussinov) fills the same triangle.

## Part 3 · Viterbi and the forward sum

The lecture's two-state machine, with numbers. `I` (island) writes C or G, each with probability $1/2$.
`B` (background) writes each of A, C, G, T with probability $1/4$. Begin in either state with $1/2$; stay $3/4$; switch $1/4$.

One function fills the table. With `max` it keeps the best labelling (Viterbi); with `sum` it adds up every labelling (forward).

In [ ]:
from fractions import Fraction as Fr

states = ["I", "B"]
begin = {"I": Fr(1, 2), "B": Fr(1, 2)}
trans = {"I": {"I": Fr(3, 4), "B": Fr(1, 4)},
         "B": {"I": Fr(1, 4), "B": Fr(3, 4)}}
emit = {"I": {"A": Fr(0), "C": Fr(1, 2), "G": Fr(1, 2), "T": Fr(0)},
        "B": {"A": Fr(1, 4), "C": Fr(1, 4), "G": Fr(1, 4), "T": Fr(1, 4)}}

def fill(x, combine):
    """V[t][k]: combine over labellings of x[0..t] that end in state k."""
    V = [{k: begin[k] * emit[k][x[0]] for k in states}]
    ptr = [{k: None for k in states}]
    for t in range(1, len(x)):
        col, back = {}, {}
        for k in states:
            cands = {j: V[t - 1][j] * trans[j][k] for j in states}
            back[k] = max(cands, key=cands.get)
            col[k] = emit[k][x[t]] * combine(cands.values())
        V.append(col)
        ptr.append(back)
    return V, ptr

def viterbi(x):
    V, ptr = fill(x, max)
    k = max(states, key=lambda s: V[-1][s])
    best = V[-1][k]
    path = [k]
    for t in range(len(x) - 1, 0, -1):
        path.append(ptr[t][path[-1]])
    return "".join(reversed(path)), best

def forward(x):
    V, _ = fill(x, sum)
    return sum(V[-1].values())

path, p = viterbi("CG")
print("CG  best labelling:", path, " probability:", p)
print("CG  total over all labellings:", forward("CG"))
assert path == "II" and p == Fr(3, 32)
assert forward("CG") == Fr(3, 32) + Fr(1, 64) + Fr(1, 64) + Fr(3, 128)

### A longer sequence

The labelling drawn in the lecture: background for A and T, island across the CG run.

In [ ]:
x = "ATCGCGA"
path, p = viterbi(x)
print(x)
print(path)
print("best labelling:", p, " total:", forward(x), " share of the total:", float(p / forward(x)))
assert path == "BBIIIIB"

**Play.** Make switching rarer (`Fr(1, 20)`) or the island less extreme (C and G $0.4$, A and T $0.1$). On `"ATCGATCGCGCGCGAT"`, where does the island start and stop?

## Questions

**Q1.** On `ACATATG` against `ACTATA`, in which row does the best path cross the middle column, and which two numbers add up there?

A1.

**Q2.** What does one Nussinov cell $N(i,j)$ store?

A2.

**Q3.** Why does a Viterbi cell read only the column before it, and what changes if $\max$ is replaced by a sum?

A3.